# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [2]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on GitHub we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [5]:
# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [6]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [7]:
cur.execute("""
SELECT
    sales.id AS sale_id,
    sales.date,
    sales.price,
    sales.house_id,
    details.bedrooms,
    details.bathrooms,
    details.sqft_living,
    details.sqft_lot,
    details.floors,
    details.waterfront,
    details.view,
    details.condition,
    details.grade,
    details.sqft_above,
    details.sqft_basement,
    details.yr_built,
    details.yr_renovated,
    details.zipcode,
    details.lat,
    details.long,
    details.sqft_living15,
    details.sqft_lot15
FROM eda.king_county_house_sales AS sales
JOIN eda.king_county_house_details AS details
    ON sales.house_id = details.id
LIMIT 10;
""")
cur.fetchall()

[(1,
  datetime.date(2014, 10, 13),
  221900.0,
  7129300520,
  3.0,
  1.0,
  1180.0,
  5650.0,
  1.0,
  None,
  0.0,
  3,
  7,
  1180.0,
  0.0,
  1955,
  0,
  98178,
  47.5112,
  -122.257,
  1340.0,
  5650.0),
 (2,
  datetime.date(2014, 12, 9),
  538000.0,
  6414100192,
  3.0,
  2.25,
  2570.0,
  7242.0,
  2.0,
  0.0,
  0.0,
  3,
  7,
  2170.0,
  400.0,
  1951,
  19910,
  98125,
  47.721,
  -122.319,
  1690.0,
  7639.0),
 (3,
  datetime.date(2015, 2, 25),
  180000.0,
  5631500400,
  2.0,
  1.0,
  770.0,
  10000.0,
  1.0,
  0.0,
  0.0,
  3,
  6,
  770.0,
  0.0,
  1933,
  None,
  98028,
  47.7379,
  -122.233,
  2720.0,
  8062.0),
 (4,
  datetime.date(2014, 12, 9),
  604000.0,
  2487200875,
  4.0,
  3.0,
  1960.0,
  5000.0,
  1.0,
  0.0,
  0.0,
  5,
  7,
  1050.0,
  910.0,
  1965,
  0,
  98136,
  47.5208,
  -122.393,
  1360.0,
  5000.0),
 (5,
  datetime.date(2015, 2, 18),
  510000.0,
  1954400510,
  3.0,
  2.0,
  1680.0,
  8080.0,
  1.0,
  0.0,
  0.0,
  3,
  8,
  1680.0,
  0.0,
  1987,
 

With `conn.close()` you can close the connection again.

In [8]:
# close the connection
conn.close()

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [9]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

In [10]:
# import the data into a pandas dataframe
query_string = """
SELECT
    sales.id AS sale_id,
    sales.date,
    sales.price,
    sales.house_id,
    details.bedrooms,
    details.bathrooms,
    details.sqft_living,
    details.sqft_lot,
    details.floors,
    details.waterfront,
    details.view,
    details.condition,
    details.grade,
    details.sqft_above,
    details.sqft_basement,
    details.yr_built,
    details.yr_renovated,
    details.zipcode,
    details.lat,
    details.long,
    details.sqft_living15,
    details.sqft_lot15
FROM eda.king_county_house_sales AS sales
JOIN eda.king_county_house_details AS details
    ON sales.house_id = details.id
;
"""
df_psycopg = pd.read_sql(query_string, conn)

C:\Users\hadid\AppData\Local\Temp\ipykernel_22004\3014029357.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_psycopg = pd.read_sql(query_string, conn)


In [11]:
df_psycopg

,sale_id,date,price,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,1,2014-10-13,221900.0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2,2014-12-09,538000.0,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,3,2015-02-25,180000.0,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,4,2014-12-09,604000.0,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,5,2015-02-18,510000.0,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,21593,2014-05-21,360000.0,263000018,3.0,2.50,1530.0,1131.0,3.0,0.0,...,8,1530.0,0.0,2009,0.0,98103,47.6993,-122.346,1530.0,1509.0
21593,21594,2015-02-23,400000.0,6600060120,4.0,2.50,2310.0,5813.0,2.0,0.0,...,8,2310.0,0.0,2014,0.0,98146,47.5107,-122.362,1830.0,7200.0
21594,21595,2014-06-23,402101.0,1523300141,2.0,0.75,1020.0,1350.0,2.0,0.0,...,7,1020.0,0.0,2009,0.0,98144,47.5944,-122.299,1020.0,2007.0
21595,21596,2015-01-16,400000.0,291310100,3.0,2.50,1600.0,2388.0,2.0,NaN,...,8,1600.0,0.0,2004,0.0,98027,47.5345,-122.069,1410.0,1287.0


In [12]:
# close the connection
conn.close()

In [13]:
df_psycopg.head()

,sale_id,date,price,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,1,2014-10-13,221900.0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2,2014-12-09,538000.0,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,3,2015-02-25,180000.0,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,4,2014-12-09,604000.0,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,5,2015-02-18,510000.0,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [14]:
# export the data to a csv-file
df_psycopg.to_csv("data/eda.csv", index=False)

In [15]:
df_psycopg.shape

(21597, 22)

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database string (a link that includes every information we entered in the conn object)

In [16]:
from sqlalchemy import create_engine

# read the database string from the .env
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [17]:
# import the data to a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales"
df_sqlalchemy = pd.read_sql(query_string, db)

In [18]:
df_sqlalchemy.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


In [19]:
query_string_details = "SELECT * FROM eda.king_county_house_details"

df_details = pd.read_sql(query_string_details, db)
df_details.head

<bound method NDFrame.head of                id  bedrooms  bathrooms  sqft_living  sqft_lot  floors  \
0         1000102       6.0       3.00       2400.0    9373.0     2.0   
1       100100050       3.0       1.00       1320.0   11090.0     1.0   
2      1001200035       3.0       1.00       1350.0    7973.0     1.5   
3      1001200050       4.0       1.50       1260.0    7248.0     1.5   
4      1003000175       3.0       1.00        980.0    7606.0     1.0   
...           ...       ...        ...          ...       ...     ...   
21415   993002177       3.0       2.50       1380.0    1547.0     3.0   
21416   993002225       3.0       2.25       1520.0    1245.0     3.0   
21417   993002247       3.0       2.25       1550.0    1469.0     3.0   
21418   993002325       2.0       1.50        950.0    4625.0     1.0   
21419   999000215       4.0       2.50       2760.0    5000.0     1.5   

       waterfront  view  condition  grade  sqft_above  sqft_basement  \
0             NaN   0

Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [20]:
query_joined = """
SELECT
    sales.id AS sale_id,
    sales.date,
    sales.price,
    sales.house_id,
    details.bedrooms,
    details.bathrooms,
    details.sqft_living,
    details.sqft_lot,
    details.floors,
    details.waterfront,
    details.view,
    details.condition,
    details.grade,
    details.sqft_above,
    details.sqft_basement,
    details.yr_built,
    details.yr_renovated,
    details.zipcode,
    details.lat,
    details.long,
    details.sqft_living15,
    details.sqft_lot15
FROM eda.king_county_house_sales AS sales
JOIN eda.king_county_house_details AS details
    ON sales.house_id = details.id
"""

df_joined = pd.read_sql(query_joined, db)

In [21]:
df_joined.shape

(21597, 22)

In [22]:
df_joined.to_csv("data/eda1.csv", index=False)

In [23]:
df_import = pd.read_csv("data/eda1.csv")

In [24]:
import pandas as pd

df = pd.read_csv("data/eda1.csv")
df.shape

(21597, 22)

In [ ]:
# export the data to a csv-file
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [ ]:
# import the data from a csv-file
df_import = pd.read_csv("data/eda.csv")